# Autonomous AI Agents: Runtime, State Graphs, Memory, Multi-Agent Routing & MCP

This notebook demonstrates the architectural building blocks of production AI agent systems:
1. **Tool Use & Function Calling**: Schema definition, runtime validation, safe execution.
2. **Agentic Loops**: Thought-Action-Observation (ReAct) with step limits and human approval for high-risk tools.
3. **Memory Systems**: Conversational sliding window short-term memory + vector/cosine long-term episodic memory.
4. **State Graphs (LangGraph Pattern)**: Explicit state machines with state nodes, conditional routing, and checkpointing.
5. **Multi-Agent Orchestration**: Hierarchical supervisor pattern directing tasks to specialized worker agents.
6. **Model Context Protocol (MCP)**: JSON-RPC 2.0 protocol handling for standard client-server tool invocation.
7. **Evaluation**: Goal completion rate, tool call precision, and trajectory efficiency.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/agents/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import agent_runtime
print('agent_runtime loaded')

## 1. Tool Registry & Schema Validation
Tools expose clear JSON-Schema-like contracts so callers can reliably invoke functions.

In [ ]:
from agent_runtime import Tool, ToolRegistry, safe_calc, validate_args

registry = ToolRegistry()
registry.register(Tool(
    name="calculator",
    description="Evaluate arithmetic expression safely",
    parameters={"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
    fn=safe_calc,
    risk="low"
))

def get_weather(location: str) -> str:
    data = {"Paris": "18C, Sunny", "Tokyo": "22C, Clear", "London": "14C, Rainy"}
    return data.get(location, "20C, Partly Cloudy")

registry.register(Tool(
    name="get_weather",
    description="Retrieve weather report for a city",
    parameters={"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]},
    fn=get_weather,
    risk="low"
))

print("Registered tools:", [s["name"] for s in registry.specs()])
ok, res = registry.call("calculator", {"expression": "14 * 3 + 2"})
print(f"Calculator output (ok={ok}):", res)

## 2. ReAct Agent Loop with Step Limits
The loop performs Action -> Observation -> Repeat until `final` or step budget is reached.

In [ ]:
from agent_runtime import run_agent

def math_policy(task, history):
    # If no tool has run yet, call calculator
    if not history:
        return {"tool": "calculator", "arguments": {"expression": "25 * 4"}}
    # If calculator succeeded, return final
    last = history[-1]
    if last.get("tool") == "calculator" and last.get("ok"):
        return {"final": f"Result is {last['observation']}"}
    return {"final": "Could not solve"}

res = run_agent(math_policy, registry, "What is 25 * 4?", max_steps=5)
print("Finished status:", res.status)
print("Final answer:", res.final)
print("Step count:", res.steps)
for t in res.trace:
    print(f" - [{t['step']}] Tool: {t.get('tool')} -> Obs: {t.get('observation')}")

## 3. Human-in-the-Loop Approval for High-Risk Tools
Tools tagged `risk='high'` pause for human approval before execution.

In [ ]:
def wipe_db(table: str) -> str:
    return f"Dropped table {table}"

registry.register(Tool(
    name="wipe_db",
    description="Dangerous table drop",
    parameters={"type": "object", "properties": {"table": {"type": "string"}}, "required": ["table"]},
    fn=wipe_db,
    risk="high"
))

def risky_policy(task, history):
    return {"tool": "wipe_db", "arguments": {"table": "users"}}

# 1. Denied execution (approval hook returns False)
res_denied = run_agent(risky_policy, registry, "Drop table", max_steps=2, approve=lambda name, args: False)
print("Denied observation:", res_denied.trace[0]["observation"])

# 2. Approved execution (approval hook returns True)
res_approved = run_agent(risky_policy, registry, "Drop table", max_steps=2, approve=lambda name, args: True)
print("Approved observation:", res_approved.trace[0]["observation"])

## 4. Short-Term & Long-Term Memory
Short-term memory stores the conversation window; long-term episodic memory retrieves top-k relevant facts.

In [ ]:
from agent_runtime import ShortTermMemory, LongTermMemory

stm = ShortTermMemory(max_items=3)
for i in range(5):
    stm.add("user" if i % 2 == 0 else "assistant", f"Message {i}")
print("STM retained:", stm.items)

ltm = LongTermMemory()
ltm.add("User prefers concise Python code without comments")
ltm.add("Database port 5432 timed out under heavy load")
ltm.add("User is located in UTC+1 timezone")

recalled = ltm.search("What is user coding preference?", k=1)
print("LTM recalled:", recalled)

## 5. State Graphs (LangGraph Pattern)
Stateful cyclic computation graph: Nodes update a shared state dict, and conditional routers steer transitions.

In [ ]:
from agent_runtime import StateGraph, END

sg = StateGraph()

def draft_node(state):
    return {"draft": "AI agents combine reasoning and actions.", "iterations": state.get("iterations", 0) + 1}

def critique_node(state):
    return {"critique": "Add mention of memory and tools."}

def revise_node(state):
    return {"draft": state["draft"] + " They use memory and external tools to accomplish goals.", "approved": True}

sg.add_node("drafter", draft_node)
sg.add_node("critic", critique_node)
sg.add_node("reviser", revise_node)

sg.set_entry("drafter")
sg.add_edge("drafter", "critic")

def route_critic(state):
    return END if state.get("approved") else "reviser"

sg.add_conditional_edges("critic", route_critic)
sg.add_edge("reviser", END)

final_state, checkpoints = sg.run({"approved": False})
print("Final draft:", final_state["draft"])
print("Final approved:", final_state["approved"])
print("Execution path:", [cp[0] for cp in checkpoints])

## 6. Multi-Agent Supervisor Pattern
A coordinator classifies intent and routes queries to specialized sub-agents.

In [ ]:
from agent_runtime import supervisor_route

workers = {
    "coder": (["python", "code", "function", "debug"], lambda q: f"[Coder] Wrote implementation for: {q}"),
    "researcher": (["literature", "papers", "survey", "transformer"], lambda q: f"[Researcher] Found 3 papers on: {q}"),
    "general": (["hello", "help", "summary"], lambda q: f"[General] Responded to: {q}")
}

print(supervisor_route("Write a python function to compute factorial", workers))
print(supervisor_route("Search recent literature on transformer scaling", workers))

## 7. Model Context Protocol (MCP) Standard
Handle tool listing and tool invocation over the standard JSON-RPC 2.0 protocol.

In [ ]:
from agent_runtime import handle_jsonrpc

# 1. MCP tools/list request
req_list = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}
resp_list = handle_jsonrpc(registry, req_list)
print("MCP tools count:", len(resp_list["result"]["tools"]))
print("Tool names:", [t["name"] for t in resp_list["result"]["tools"]])

# 2. MCP tools/call request
req_call = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {"name": "calculator", "arguments": {"expression": "2 ** 10"}}
}
resp_call = handle_jsonrpc(registry, req_call)
print("MCP tool execution result:", resp_call["result"])

## 8. Agent Evaluation
Benchmarking agent runs across accuracy, tool selection, and trajectory length.

In [ ]:
from agent_runtime import evaluate_agent

cases = [
    {
        "task": "25 * 4",
        "check": lambda ans: ans is not None and "100" in ans,
        "expected_tools": ["calculator"]
    },
    {
        "task": "10 + 5",
        "check": lambda ans: ans is not None and "15" in ans,
        "expected_tools": ["calculator"]
    }
]

metrics = evaluate_agent(cases, lambda task: run_agent(math_policy, registry, task))
for k, v in metrics.items():
    print(f"{k:25s}: {v:.2f}")